# YOLOv8 Orchid Stage Detection - Google Colab Training

Run this notebook in Google Colab with GPU enabled.

Important: Colab cannot directly read your local `D:` drive. Put the dataset zip into Colab first. Use one of these methods:

1. Upload `Orchid Stage Detection new.v2i.yolov8.zip` into Colab's left `Files` panel so it appears under `/content`.
2. Or upload the zip to Google Drive `MyDrive`.

In Colab select: `Runtime > Change runtime type > T4 GPU`.

In [6]:
%pip install -q ultralytics

from ultralytics import YOLO
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Colab, select Runtime > Change runtime type > T4 GPU, then run again.')

DEVICE = 0
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [7]:
from pathlib import Path
import shutil
import zipfile

try:
    from google.colab import drive
except Exception:
    drive = None

EXTRACT_DIR = Path('/content/orchid_yolov8')

zip_candidates = [
    Path('/content/Orchid Stage Detection new.v2i.yolov8.zip'),
    Path('/content/YOLOv8.zip'),
]
zip_candidates.extend(sorted(Path('/content').glob('*.zip')))

zip_path = next((path for path in zip_candidates if path.exists()), None)

if zip_path is None and drive is not None:
    try:
        drive.mount('/content/drive', force_remount=False)
        drive_candidates = [
            Path('/content/drive/MyDrive/Orchid Stage Detection new.v2i.yolov8.zip'),
            Path('/content/drive/MyDrive/YOLOv8.zip'),
        ]
        drive_candidates.extend(sorted(Path('/content/drive/MyDrive').glob('*.zip')))
        zip_path = next((path for path in drive_candidates if path.exists()), None)
    except Exception as exc:
        print('Google Drive mount failed, continuing without Drive.')
        print('Drive error:', exc)

if zip_path is None:
    raise FileNotFoundError(
        'Dataset zip not found. In Colab, open the left Files panel and upload '
        'Orchid Stage Detection new.v2i.yolov8.zip so it appears under /content, then run this cell again.'
    )

print('Using dataset zip:', zip_path)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print('Extracted dataset to:', EXTRACT_DIR)
for path in sorted([p for p in EXTRACT_DIR.rglob('*') if p.is_dir()])[:50]:
    print(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using dataset zip: /content/drive/MyDrive/Orchid Stage Detection new.v2i.yolov8.zip
Extracted dataset to: /content/orchid_yolov8
/content/orchid_yolov8/test
/content/orchid_yolov8/test/images
/content/orchid_yolov8/test/labels
/content/orchid_yolov8/train
/content/orchid_yolov8/train/images
/content/orchid_yolov8/train/labels
/content/orchid_yolov8/valid
/content/orchid_yolov8/valid/images
/content/orchid_yolov8/valid/labels


Set the dataset path. If your zip contains one top-level `YOLOv8` folder, the first option is correct. If the zip extracts `train`, `valid`, `test`, and `data.yaml` directly into `/content/orchid_yolov8`, use the second option instead.

In [8]:
from pathlib import Path

candidate_yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
if not candidate_yaml_files:
    raise FileNotFoundError('data.yaml was not found after unzipping. Check that YOLOv8.zip contains the dataset folder.')

DATA_YAML_PATH = candidate_yaml_files[0]
DATASET_DIR = DATA_YAML_PATH.parent
DATA_YAML = str(DATA_YAML_PATH)

required_dirs = [
    DATASET_DIR / 'train' / 'images',
    DATASET_DIR / 'train' / 'labels',
    DATASET_DIR / 'valid' / 'images',
    DATASET_DIR / 'valid' / 'labels',
    DATASET_DIR / 'test' / 'images',
    DATASET_DIR / 'test' / 'labels',
]
missing_dirs = [str(path) for path in required_dirs if not path.exists()]
if missing_dirs:
    raise FileNotFoundError('Missing dataset folders: ' + ', '.join(missing_dirs))

print('DATASET_DIR:', DATASET_DIR)
print('DATA_YAML:', DATA_YAML)
print('data.yaml exists:', DATA_YAML_PATH.exists())

DATASET_DIR: /content/orchid_yolov8
DATA_YAML: /content/orchid_yolov8/data.yaml
data.yaml exists: True


In [9]:
from collections import Counter
from pathlib import Path

if 'EXTRACT_DIR' not in globals():
    EXTRACT_DIR = Path('/content/orchid_yolov8')

if 'DATA_YAML' not in globals() or 'DATASET_DIR' not in globals():
    candidate_yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
    if not candidate_yaml_files:
        raise FileNotFoundError(
            'Dataset is not extracted yet. Run the upload/unzip cell first and wait until it prints: Extracted dataset to: /content/orchid_yolov8'
        )
    DATA_YAML_PATH = candidate_yaml_files[0]
    DATASET_DIR = DATA_YAML_PATH.parent
    DATA_YAML = str(DATA_YAML_PATH)

names = ['Bud_formation', 'Flowering', 'Mature_Cane', 'Seedling', 'Vegetative']
root = Path(DATASET_DIR)

print('Checking dataset at:', root)
print('Using data.yaml:', DATA_YAML)

for split in ['train', 'valid', 'test']:
    split_dir = root / split
    image_dir = split_dir / 'images'
    label_dir = split_dir / 'labels'
    image_files = list(image_dir.glob('*.jpg')) + list(image_dir.glob('*.jpeg')) + list(image_dir.glob('*.png'))
    label_files = list(label_dir.glob('*.txt'))
    counts = Counter()
    missing_labels = []

    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f'Missing folder for {split}: {image_dir} or {label_dir}')

    for image_file in image_files:
        if not (label_dir / f'{image_file.stem}.txt').exists():
            missing_labels.append(image_file.name)

    for label_file in label_files:
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1

    print(f'\n{split}: {len(image_files)} images, {len(label_files)} labels')
    print('  missing label files:', len(missing_labels))
    for class_id, class_name in enumerate(names):
        print(f'  {class_id}: {class_name}: {counts[class_id]}')

    if not image_files:
        raise RuntimeError(f'No images found in {image_dir}. Check that you uploaded the correct YOLOv8 dataset zip file.')
    if missing_labels:
        raise RuntimeError(f'Some {split} images do not have matching label files.')

Checking dataset at: /content/orchid_yolov8
Using data.yaml: /content/orchid_yolov8/data.yaml

train: 420 images, 420 labels
  missing label files: 0
  0: Bud_formation: 21
  1: Flowering: 161
  2: Mature_Cane: 77
  3: Seedling: 42
  4: Vegetative: 126

valid: 3 images, 3 labels
  missing label files: 0
  0: Bud_formation: 0
  1: Flowering: 0
  2: Mature_Cane: 0
  3: Seedling: 0
  4: Vegetative: 3

test: 12 images, 12 labels
  missing label files: 0
  0: Bud_formation: 2
  1: Flowering: 3
  2: Mature_Cane: 2
  3: Seedling: 4
  4: Vegetative: 1


In [10]:
from pathlib import Path

if 'DEVICE' not in globals():
    DEVICE = 0 if torch.cuda.is_available() else 'cpu'

if 'DATA_YAML' not in globals():
    EXTRACT_DIR = Path('/content/orchid_yolov8')
    candidate_yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
    if not candidate_yaml_files:
        raise FileNotFoundError(
            'Cannot start training because data.yaml was not found. Run the upload/unzip cell first and make sure YOLOv8.zip is uploaded completely.'
        )
    DATA_YAML_PATH = candidate_yaml_files[0]
    DATASET_DIR = DATA_YAML_PATH.parent
    DATA_YAML = str(DATA_YAML_PATH)

RUN_NAME = 'yolov8s_512_100epochs'
RUN_PROJECT = '/content/orchid_yolov8_runs'

print('Training with:', DATA_YAML)
print('Saving runs to:', f'{RUN_PROJECT}/{RUN_NAME}')

model = YOLO('yolov8s.pt')

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=512,
    batch=8,
    patience=30,
    device=DEVICE,
    project=RUN_PROJECT,
    name=RUN_NAME,
    exist_ok=True
)

BEST_MODEL_PATH = f'{RUN_PROJECT}/{RUN_NAME}/weights/best.pt'
print('Best model saved at:', BEST_MODEL_PATH)

Training with: /content/orchid_yolov8/data.yaml
Saving runs to: /content/orchid_yolov8_runs/yolov8s_512_100epochs
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/orchid_yolov8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=

In [11]:
from pathlib import Path

if 'BEST_MODEL_PATH' not in globals():
    RUN_NAME = 'yolov8s_512_100epochs'
    RUN_PROJECT = '/content/orchid_yolov8_runs'
    BEST_MODEL_PATH = f'{RUN_PROJECT}/{RUN_NAME}/weights/best.pt'

if not Path(BEST_MODEL_PATH).exists():
    raise FileNotFoundError(f'Best model not found: {BEST_MODEL_PATH}. Run the training cell first.')

best_model = YOLO(BEST_MODEL_PATH)

val_metrics = best_model.val(data=DATA_YAML, split='val', imgsz=512, device=DEVICE)
test_metrics = best_model.val(data=DATA_YAML, split='test', imgsz=512, device=DEVICE)

val_map50 = float(val_metrics.box.map50)
val_map = float(val_metrics.box.map)
val_precision = float(val_metrics.box.mp)
val_recall = float(val_metrics.box.mr)

test_map50 = float(test_metrics.box.map50)
test_map = float(test_metrics.box.map)
test_precision = float(test_metrics.box.mp)
test_recall = float(test_metrics.box.mr)

combined_map50 = (val_map50 + test_map50) / 2
combined_map = (val_map + test_map) / 2
combined_precision = (val_precision + test_precision) / 2
combined_recall = (val_recall + test_recall) / 2

print('\nValidation metrics')
print(f'  Precision : {val_precision:.4f}')
print(f'  Recall    : {val_recall:.4f}')
print(f'  mAP50     : {val_map50:.4f}')
print(f'  mAP50-95  : {val_map:.4f}')

print('\nTest metrics')
print(f'  Precision : {test_precision:.4f}')
print(f'  Recall    : {test_recall:.4f}')
print(f'  mAP50     : {test_map50:.4f}')
print(f'  mAP50-95  : {test_map:.4f}')

print('\nCombined (validation + test average)')
print(f'  Precision : {combined_precision:.4f}')
print(f'  Recall    : {combined_recall:.4f}')
print(f'  mAP50     : {combined_map50:.4f}')
print(f'  mAP50-95  : {combined_map:.4f}')

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 951.5±454.0 MB/s, size: 36.0 KB)
val: Scanning /content/orchid_yolov8/valid/labels.cache... 3 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3/3 1.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 12.4it/s 0.1s
                   all          3          3        0.5          1       0.83      0.507
            Vegetative          3          3        0.5          1       0.83      0.507
Speed: 0.6ms preprocess, 17.2ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /content/runs/detect/val
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 780.6±287.4 MB/s, size: 38.0 KB)
val: Scanning /content/orc

In [12]:
if 'best_model' not in globals():
    best_model = YOLO(BEST_MODEL_PATH)

PRED_PROJECT = '/content/orchid_yolov8_predictions'
PRED_NAME = 'test_predictions'

pred_results = best_model.predict(
    source=f'{DATASET_DIR}/test/images',
    imgsz=512,
    conf=0.25,
    save=True,
    project=PRED_PROJECT,
    name=PRED_NAME,
    exist_ok=True
)

PRED_DIR = f'{PRED_PROJECT}/{PRED_NAME}'
print('Predictions saved at:', PRED_DIR)


image 1/12 /content/orchid_yolov8/test/images/1-2-_jpeg.rf.6934e3d55dd1da958de4b7b212cf6a65.jpg: 512x512 1 Vegetative, 11.0ms
image 2/12 /content/orchid_yolov8/test/images/20260502_161648_jpg.rf.22bb6166c3bbb7f48eaaac970b839b05.jpg: 512x512 2 Vegetatives, 11.0ms
image 3/12 /content/orchid_yolov8/test/images/27638_jpeg.rf.186562bd638d326d059543ffde5ae522.jpg: 512x512 1 Flowering, 11.0ms
image 4/12 /content/orchid_yolov8/test/images/2_jpeg.rf.fe7ec1dc9cddfb1a918c26f47a9a2064.jpg: 512x512 1 Bud_formation, 1 Vegetative, 11.0ms
image 5/12 /content/orchid_yolov8/test/images/3_jpg.rf.2575565aa2b3ccb92aa50e3c50348fa8.jpg: 512x512 1 Flowering, 1 Vegetative, 11.1ms
image 6/12 /content/orchid_yolov8/test/images/5-2-_jpg.rf.b8cab6b81856f0ae8e3cbefeb5404f4a.jpg: 512x512 1 Flowering, 1 Vegetative, 11.0ms
image 7/12 /content/orchid_yolov8/test/images/5_jpg.rf.d4a6a295bf3b8ace57df33cde71a6080.jpg: 512x512 1 Flowering, 11.0ms
image 8/12 /content/orchid_yolov8/test/images/6_jpg.rf.a11fbb082fe5ead4abc71

## Save Trained Output for This Local Project

Colab runs on Google's computer, so it cannot write directly to your local `D:` drive folder. This cell creates a zip file containing the trained model and results, then downloads it. After download, extract it into this local project folder, for example into `trained_models/`.

In [1]:
from google.colab import files
from pathlib import Path
import zipfile

if 'RUN_NAME' not in globals():
    RUN_NAME = 'yolov8s_512_100epochs'
    RUN_PROJECT = '/content/orchid_yolov8_runs'
    BEST_MODEL_PATH = f'{RUN_PROJECT}/{RUN_NAME}/weights/best.pt'

run_dir = Path(RUN_PROJECT) / RUN_NAME
pred_dir = Path(globals().get('PRED_DIR', '/content/orchid_yolov8_predictions/test_predictions'))
export_zip = Path('/content/orchid_yolov8_trained_output.zip')

if not Path(BEST_MODEL_PATH).exists():
    raise FileNotFoundError(f'best.pt not found: {BEST_MODEL_PATH}. Train the model before running this cell.')

if export_zip.exists():
    export_zip.unlink()

with zipfile.ZipFile(export_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
    for folder in [run_dir, pred_dir]:
        if folder.exists():
            for file_path in folder.rglob('*'):
                if file_path.is_file():
                    zip_file.write(file_path, arcname=file_path.relative_to(folder.parent))
        else:
            print('Skipping missing folder:', folder)

print('Created:', export_zip)
print('Zip size MB:', round(export_zip.stat().st_size / (1024 * 1024), 2))
files.download(str(export_zip))

ModuleNotFoundError: No module named 'google'